In [1]:
# MODIFIED FROM: phase3-osm-feature-extraction.ipynb
# CHANGES:
#   - Input: prediction_points.csv instead of DHS GPS shapefile
#   - Key column: PointID instead of DHSCLUST
#   - Buffer: Urban_Rural column ('U'=2km, 'R'=5km)
#   - VIIRS merge: uses viirs_ntl_2025_prediction_points.csv
#     with PointID key
#   - Output: static_osm_features_2025_prediction_points.csv
#   - OSM feature schema is identical to training (20 columns)
#     so it can be passed directly to the LSTM model

import geopandas as gpd
import pandas as pd
import numpy as np
import os
import warnings
from shapely.geometry import Point
from datetime import datetime
warnings.filterwarnings('ignore')

print('Imports ready.')

Imports ready.


In [2]:
# ==========================================
# 1. SETUP PATHS
# ==========================================
# CHANGED:
#   - prediction_points_csv replaces dhs_shp_path
#   - viirs_csv_path points to 2025 prediction points VIIRS
#   - output_csv is a new file for inference
# UNCHANGED:
#   - base_osm_dir (same OSM shapefile as training)

# OSM shapefile folder (same as training)
base_osm_dir = '/Users/ruben/Desktop/Thesis/TrainingData/PH_OSM.shp'

# CHANGED: prediction points CSV instead of DHS GPS shapefile
prediction_points_csv = '/Users/ruben/Desktop/Thesis/2025Data/prediction_points.csv'

# CHANGED: 2025 VIIRS for prediction points (output of phase5-3)
viirs_csv_path = '/Users/ruben/Desktop/Thesis/2025Data/viirs_ntl_2025_prediction_points.csv'

# CHANGED: output file for inference
output_csv = '/Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points.csv'

# Verify all input files exist
for label, path in [
    ('OSM dir',           base_osm_dir),
    ('Prediction points', prediction_points_csv),
    ('VIIRS 2025',        viirs_csv_path)
]:
    status = '\u2713' if os.path.exists(path) else '\u2717 MISSING'
    print(f'  {status}  {label}: {path}')

  ✓  OSM dir: /Users/ruben/Desktop/Thesis/TrainingData/PH_OSM.shp
  ✓  Prediction points: /Users/ruben/Desktop/Thesis/2025Data/prediction_points.csv
  ✓  VIIRS 2025: /Users/ruben/Desktop/Thesis/2025Data/viirs_ntl_2025_prediction_points.csv


In [3]:
# ==========================================
# 2. LOAD DATA
# ==========================================
# CHANGED: loads prediction_points.csv and converts to
# GeoDataFrame from Latitude/Longitude columns.
# The resulting gdf_points has the same structure as the
# original gdf_dhs but keyed on PointID and Urban_Rural.
# UNCHANGED: OSM shapefile loading logic.

print('Loading prediction points CSV...')
df_pts = pd.read_csv(prediction_points_csv)
print(f'Prediction points: {len(df_pts)}')
print(f'Columns: {list(df_pts.columns)}')
print(f'Province distribution:')
print(df_pts['Province'].value_counts().to_string())

# Convert to GeoDataFrame using Latitude/Longitude
geometry    = [Point(xy) for xy in
               zip(df_pts['Longitude'], df_pts['Latitude'])]
gdf_points  = gpd.GeoDataFrame(
    df_pts, geometry=geometry, crs='EPSG:4326')
print(f'\nGeoDataFrame created: {len(gdf_points)} points')

# Load OSM files (unchanged from training)
print('\nLoading OSM shapefiles...')
try:
    gdf_roads = gpd.read_file(
        os.path.join(base_osm_dir, 'gis_osm_roads_free_1.shp'))
    gdf_bldgs = gpd.read_file(
        os.path.join(base_osm_dir,
                     'gis_osm_buildings_a_free_1.shp'))
    gdf_pois  = gpd.read_file(
        os.path.join(base_osm_dir, 'gis_osm_pois_free_1.shp'))
    print('\u2713 Loaded Roads, Buildings, and POIs.')
    print(f'  Roads     : {len(gdf_roads):,} features')
    print(f'  Buildings : {len(gdf_bldgs):,} features')
    print(f'  POIs      : {len(gdf_pois):,} features')
except Exception as e:
    print(f'Error loading OSM files: {e}')
    print('Ensure the OSM shapefile folder exists.')

Loading prediction points CSV...
Prediction points: 1240
Columns: ['PointID', 'Latitude', 'Longitude', 'Municipality', 'Province', 'Region', 'Urban_Rural', 'source', 'DHSCLUST', 'PSGC', 'PSA_Poverty_Rate']
Province distribution:
Province
Zamboanga del Norte    239
Davao Oriental         194
NCR                    149
Ilocos Norte           119
Kalinga                116
Benguet                107
Pampanga                84
Maguindanao del Sur     83
Aklan                   61
Basilan                 44
Tawi-Tawi               44

GeoDataFrame created: 1240 points

Loading OSM shapefiles...
✓ Loaded Roads, Buildings, and POIs.
  Roads     : 1,584,797 features
  Buildings : 11,730,420 features
  POIs      : 168,727 features


In [4]:
# ==========================================
# 3. REPROJECT AND BUFFER
# ==========================================
# CHANGED: buffer logic now uses Urban_Rural column from
# prediction_points.csv instead of URBAN_RURA from shapefile.
#   Urban_Rural == 'U' -> 2,000m (NCR)
#   Urban_Rural == 'R' -> 5,000m (all other provinces)
# This exactly mirrors the training buffer protocol.
# UNCHANGED: target CRS, buffer distances.

target_crs = 'EPSG:32651'

print('Reprojecting to EPSG:32651...')
gdf_points = gdf_points.to_crs(target_crs)
gdf_roads  = gdf_roads.to_crs(target_crs)
gdf_bldgs  = gdf_bldgs.to_crs(target_crs)
gdf_pois   = gdf_pois.to_crs(target_crs)

# CHANGED: use Urban_Rural column (was URBAN_RURA)
print('Creating adaptive buffers...')
def get_buffer(row):
    if str(row.get('Urban_Rural', 'R')).upper() == 'U':
        return row.geometry.buffer(2000)  # 2km for Urban (NCR)
    else:
        return row.geometry.buffer(5000)  # 5km for Rural

gdf_points['buffer_geom'] = gdf_points.apply(
    get_buffer, axis=1)

# Verify buffer sizes
urban_count = (gdf_points['Urban_Rural'].str.upper() == 'U').sum()
rural_count = (gdf_points['Urban_Rural'].str.upper() == 'R').sum()
print(f'Urban points (2km buffer): {urban_count}')
print(f'Rural points (5km buffer): {rural_count}')
print('\u2713 Buffers created')

Reprojecting to EPSG:32651...
Creating adaptive buffers...
Urban points (2km buffer): 149
Rural points (5km buffer): 1091
✓ Buffers created


In [5]:
# ==========================================
# 4. FEATURE ENGINEERING LOOP
# ==========================================
# CHANGED:
#   - Iterates over gdf_points (from CSV) instead of gdf_dhs
#   - Uses PointID instead of DHSCLUST as the record key
#   - feature_row keyed on 'PointID' not 'DHSCLUST'
# UNCHANGED: all road, building, POI extraction logic,
#   spatial indexing, clipping, and feature categories.
#   The 20 OSM feature columns are identical to training.

print('Extracting OSM features for prediction points...')
results = []

# Build spatial indexes
road_sindex = gdf_roads.sindex
bldg_sindex = gdf_bldgs.sindex
poi_sindex  = gdf_pois.sindex

road_types = {
    'Main_Roads'     : ['motorway', 'trunk',
                        'primary', 'primary_link'],
    'Secondary_Roads': ['secondary', 'tertiary'],
    'Local_Roads'    : ['residential', 'living_street',
                        'unclassified', 'service'],
    'Tracks'         : ['track', 'path']
}
target_bldgs = ['residential', 'commercial',
                'industrial', 'school', 'hospital']
wealth_pois  = ['bank', 'hotel', 'fast_food',
                'convenience', 'school', 'hospital']

# CHANGED: iterate over gdf_points, use PointID
for loop_idx, (idx, row) in enumerate(
        gdf_points.iterrows()):

    point_id = row['PointID']    # CHANGED from DHSCLUST
    buffer   = row['buffer_geom']
    n_done   = loop_idx + 1

    # ── ROADS ──────────────────────────────────────────
    possible_roads = gdf_roads.iloc[
        list(road_sindex.intersection(buffer.bounds))]
    precise_roads  = possible_roads[
        possible_roads.intersects(buffer)]

    road_stats = {}
    if len(precise_roads) > 0:
        clipped = precise_roads.geometry.intersection(buffer)
        road_stats['Total_Road_Length'] = (
            clipped.length.sum() / 1000.0)
        for r_cat, r_cls in road_types.items():
            mask = precise_roads['fclass'].isin(r_cls)
            road_stats[f'{r_cat}_Length'] = (
                clipped[mask].length.sum() / 1000.0)
    else:
        road_stats = {f'{k}_Length': 0
                      for k in road_types}
        road_stats['Total_Road_Length'] = 0

    # ── BUILDINGS ──────────────────────────────────────
    possible_bldgs = gdf_bldgs.iloc[
        list(bldg_sindex.intersection(buffer.bounds))]
    precise_bldgs  = possible_bldgs[
        possible_bldgs.intersects(buffer)]

    bldg_stats = {'Total_Bldg_Count': len(precise_bldgs)}
    if len(precise_bldgs) > 0:
        type_col = ('type' if 'type' in precise_bldgs.columns
                    else 'fclass')
        counts   = precise_bldgs[type_col].value_counts()
        for t in target_bldgs:
            bldg_stats[f'Bldg_{t}_Count'] = counts.get(t, 0)
        bldg_stats['Total_Bldg_Area'] = (
            precise_bldgs.area.sum())
    else:
        for t in target_bldgs:
            bldg_stats[f'Bldg_{t}_Count'] = 0
        bldg_stats['Total_Bldg_Area'] = 0

    # ── POIs ───────────────────────────────────────────
    possible_pois = gdf_pois.iloc[
        list(poi_sindex.intersection(buffer.bounds))]
    precise_pois  = possible_pois[
        possible_pois.intersects(buffer)]

    poi_stats = {'Total_POI_Count': len(precise_pois)}
    if len(precise_pois) > 0:
        counts = precise_pois['fclass'].value_counts()
        for p in wealth_pois:
            poi_stats[f'POI_{p}_Count'] = counts.get(p, 0)
    else:
        for p in wealth_pois:
            poi_stats[f'POI_{p}_Count'] = 0

    # ── COMBINE ────────────────────────────────────────
    # CHANGED: key is PointID instead of DHSCLUST
    feature_row = {'PointID': point_id}
    feature_row.update(road_stats)
    feature_row.update(bldg_stats)
    feature_row.update(poi_stats)
    results.append(feature_row)

    # Progress + checkpoint
    if n_done % 50 == 0 or n_done == len(gdf_points):
        print(f'  [{datetime.now().strftime("%H:%M:%S")}] '
              f'{n_done}/{len(gdf_points)} points done')

    if n_done % 200 == 0:
        cp = output_csv.replace('.csv',
                                f'_checkpoint_{n_done}.csv')
        pd.DataFrame(results).to_csv(cp, index=False)
        print(f'  Checkpoint saved: {cp}')

print(f'\n\u2713 Extraction complete. {len(results)} records.')

Extracting OSM features for prediction points...
  [03:34:07] 50/1240 points done
  [03:34:07] 100/1240 points done
  [03:34:10] 150/1240 points done
  [03:34:12] 200/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_checkpoint_200.csv
  [03:34:13] 250/1240 points done
  [03:34:13] 300/1240 points done
  [03:34:13] 350/1240 points done
  [03:34:13] 400/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_checkpoint_400.csv
  [03:34:13] 450/1240 points done
  [03:34:13] 500/1240 points done
  [03:34:13] 550/1240 points done
  [03:34:14] 600/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/2025Data/static_osm_features_2025_prediction_points_checkpoint_600.csv
  [03:34:15] 650/1240 points done
  [03:34:15] 700/1240 points done
  [03:34:16] 750/1240 points done
  [03:34:16] 800/1240 points done
  Checkpoint saved: /Users/ruben/Desktop/Thesis/202

In [6]:
# ==========================================
# 5. MERGE VIIRS AND SAVE
# ==========================================
# CHANGED:
#   - Merges on PointID instead of DHSCLUST
#   - Loads viirs_ntl_2025_prediction_points.csv
#     (output of phase5-3, keyed on PointID)
#   - Output column order matches training static features
#     exactly so it can be fed directly to the scaler and model
# UNCHANGED: VIIRS_Median column name, merge strategy,
#   fillna logic, final column schema.

# Build OSM DataFrame
df_osm = pd.DataFrame(results)
df_osm['PointID'] = df_osm['PointID'].astype(int)

print(f'OSM features extracted : {len(df_osm)} points')
print(f'OSM columns            : {list(df_osm.columns)}')

# Load 2025 VIIRS
print(f'\nLoading VIIRS 2025 from: {viirs_csv_path}')
df_viirs = pd.read_csv(viirs_csv_path)
df_viirs['PointID'] = df_viirs['PointID'].astype(int)

# Rename VIIRS value column if needed
if 'VIIRS_Median' not in df_viirs.columns:
    for col in df_viirs.columns:
        if col.lower() in ('median', 'ntl_value', 'avg_rad',
                           'viirs_median'):
            df_viirs = df_viirs.rename(
                columns={col: 'VIIRS_Median'})
            break

df_viirs_clean = df_viirs[['PointID', 'VIIRS_Median']].copy()

# Coverage check before merge
osm_ids   = set(df_osm['PointID'])
viirs_ids = set(df_viirs_clean['PointID'])
missing   = osm_ids - viirs_ids
if missing:
    print(f'WARNING: {len(missing)} points have no VIIRS data.')
    print(f'  Will be filled with 0.')
else:
    print(f'\u2713 Full VIIRS coverage for all {len(osm_ids)} points.')

# Merge OSM + VIIRS
df_final = pd.merge(
    df_osm,
    df_viirs_clean,
    on='PointID',
    how='left'
)
df_final['VIIRS_Median'] = df_final['VIIRS_Median'].fillna(0)
df_final = df_final.sort_values('PointID').reset_index(drop=True)

# Verify column schema matches training
EXPECTED_OSM_COLS = [
    'Total_Road_Length', 'Main_Roads_Length',
    'Secondary_Roads_Length', 'Local_Roads_Length',
    'Tracks_Length', 'Total_Bldg_Count',
    'Bldg_residential_Count', 'Bldg_commercial_Count',
    'Bldg_industrial_Count', 'Bldg_school_Count',
    'Bldg_hospital_Count', 'Total_Bldg_Area',
    'Total_POI_Count', 'POI_bank_Count',
    'POI_hotel_Count', 'POI_fast_food_Count',
    'POI_convenience_Count', 'POI_school_Count',
    'POI_hospital_Count', 'VIIRS_Median'
]
missing_cols = [c for c in EXPECTED_OSM_COLS
                if c not in df_final.columns]
if missing_cols:
    print(f'WARNING: Missing columns: {missing_cols}')
else:
    print(f'\u2713 All 20 static feature columns present.')
    print(f'  Schema matches training static_osm_features_full.csv')

# Save
df_final.to_csv(output_csv, index=False)

print(f'\n{"="*50}')
print('EXTRACTION SUMMARY')
print(f'{"="*50}')
print(f'Points extracted : {len(df_final)}')
print(f'Total columns    : {len(df_final.columns)}')
print(f'  PointID + 20 static OSM features')
print(f'Missing values   : {df_final.isnull().sum().sum()}')
print(f'VIIRS zeros      : {(df_final["VIIRS_Median"]==0).sum()}')
print(f'\nSample:')
print(df_final[['PointID'] + EXPECTED_OSM_COLS[:5]].head())
print(f'\nSaved to: {output_csv}')
print(f'\nAll columns:')
print(list(df_final.columns))
print(f'\nNext: run phase6-build-inference-features to merge')
print(f'this file with dynamic CNN features for LSTM inference.')

OSM features extracted : 1240 points
OSM columns            : ['PointID', 'Total_Road_Length', 'Main_Roads_Length', 'Secondary_Roads_Length', 'Local_Roads_Length', 'Tracks_Length', 'Total_Bldg_Count', 'Bldg_residential_Count', 'Bldg_commercial_Count', 'Bldg_industrial_Count', 'Bldg_school_Count', 'Bldg_hospital_Count', 'Total_Bldg_Area', 'Total_POI_Count', 'POI_bank_Count', 'POI_hotel_Count', 'POI_fast_food_Count', 'POI_convenience_Count', 'POI_school_Count', 'POI_hospital_Count']

Loading VIIRS 2025 from: /Users/ruben/Desktop/Thesis/2025Data/viirs_ntl_2025_prediction_points.csv
✓ Full VIIRS coverage for all 1240 points.
✓ All 20 static feature columns present.
  Schema matches training static_osm_features_full.csv

EXTRACTION SUMMARY
Points extracted : 1240
Total columns    : 21
  PointID + 20 static OSM features
Missing values   : 0
VIIRS zeros      : 0

Sample:
   PointID  Total_Road_Length  Main_Roads_Length  Secondary_Roads_Length  \
0        1          31.009198           0.00000